# Incubation Portfolio Optimizer
## A Policy-Aligned Pipeline for Evidence-Based Venture Selection

This notebook implements a reproducible two-stage pipeline for incubator portfolio design
under government-mandated multi-objective constraints:

**Stage 1 — Predictive Modeling**
- Ridge-penalized logistic regression per outcome, penalty selected by LOOCV (log-loss)
- Out-of-sample performance: AUC, Log-Loss, Brier score via LOOCV
- Permutation tests for significance of predictive signal
- Calibration assessment: reliability curves, Brier decomposition, isotonic post-calibration
- Average Marginal Effects (AMEs) with bootstrap 95% CIs

**Stage 2 — Portfolio Optimization**
- Per-venture objective coefficients from AMEs
- Multi-Objective Binary Integer Program (MO-BIP) solved with PuLP/CBC
- ε-constraint method to trace Pareto frontiers under capacity and composition constraints
- Monte Carlo uncertainty propagation (AME + baseline sampling)
- Evidence-weighted objectives (optional: scale by permutation p-value confidence)

**Outcomes** (binary, mapped to government KPI categories):

| Variable | Policy KPI |
|---|---|
| `ip` | Intellectual property / technology transfer |
| `hub` | Regional anchoring / science park affiliation |
| `rev_high` | Revenue / commercialization impact |
| `team_growth` | Employment generation |
| `stage_growth` | Technological stage advancement |

**Predictors**:

| Variable | Type | Description |
|---|---|---|
| `tech_digital` | binary | Tech domain: 1=digital/platform/Industry-4.0, 0=bio/agri |
| `founders` | integer | Number of founding partners |
| `incub_years` | numeric | Incubation duration (years) |
| `stage_in` | ordinal (1–4) | Technology stage at entry: 1=Ideation → 4=Scale-up |
| `team_size_in` | binary | Team size bracket at entry (0=<5, 1=5–10 employees) |
| `stage_adv` | binary | Advanced stage at entry: `stage_in` > 2 |

**Usage**: Set `DATA_PATH` in *Section 1* to your cleaned dataset (`data/venture_cohort.csv`).
Adjust `OUTCOME_SPECS` to match your covariates.
All random operations are seeded for full reproducibility.

---
*Reference*: [Author(s) omitted for review]. Designing Incubation Portfolios Aligned
with Public Policy Objectives. *Technovation* (under review).


## Section 1 — Setup & Configuration

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
# If this cell raises ModuleNotFoundError, install dependencies first:
#   pip install -r requirements.txt
import warnings, json
warnings.simplefilter("ignore", UserWarning)
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import LeaveOneOut, StratifiedKFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    from sklearn.calibration import calibration_curve, CalibratedClassifierCV
    from scipy.special import expit
    import pulp
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        f"Missing package: {{_e}}. Run: pip install -r requirements.txt"
    ) from _e

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print("All imports OK.")

In [ ]:
from pathlib import Path  # ensure Path is available even if Cell 1 was not run


# ── File paths (update DATA_PATH to your cleaned CSV) ────────────────────────
DATA_PATH      = Path("data/venture_cohort.csv")        # output of data_preparation step
DICT_PATH      = Path("data/venture_cohort_dictionary.json")
OUTPUTS_DIR    = Path("outputs")
OUTPUTS_DIR.mkdir(exist_ok=True)
# ── Reproducibility seeds ─────────────────────────────────────────────────────
SEED_PERM  = 42     # permutation tests
SEED_BOOT  = 123    # pairs bootstrap
SEED_MC    = 42     # Monte Carlo
# ── Hyperparameters ───────────────────────────────────────────────────────────
C_GRID    = np.logspace(-3, 3, 25)    # ridge penalty search grid (C = 1/lambda)
B_PERM    = 2_000                      # permutation draws
R_BOOT    = 2_000                      # bootstrap resamples
MC_DRAWS  = 2_000                      # Monte Carlo draws
CAPACITY  = 8                          # incubator slot capacity (K)
ALPHAS    = [0.4, 0.6, 0.8, 1.0]      # Expand-vs-Admit gain scaling
EPS_POINTS = 6                             # ε values per secondary (6^4=1296 instances)
EPS_VEC   = np.linspace(0.05, 0.95, 19)  # ε grid for Monte Carlo scenario sweep
print("Configuration loaded.")
print(f"  Data : {DATA_PATH}")
print(f"  B_PERM={B_PERM}, R_BOOT={R_BOOT}, MC_DRAWS={MC_DRAWS}, K={CAPACITY}")


In [ ]:
# ── Outcome specification: which predictors enter each model ─────────────────
#    bin_cols          : binary predictors (0/1), kept as-is
#    num_cols          : continuous/ordinal predictors, standardized (mean=0, sd=1)
#    cat_cols          : categorical predictors, one-hot encoded (drop_first=True)
#    interact_num_cat  : list of (num_col, cat_col) pairs to add num*dummy interaction columns
#
#    Variable names match the columns in venture_cohort.csv.
#    Adapt this dict to your own dataset's column names.
OUTCOME_SPECS = {
    "ip":           {"bin_cols": ["tech_digital"],              "num_cols": ["founders"],    "cat_cols": [],           "interact_num_cat": []},
    "hub":          {"bin_cols": [],                            "num_cols": ["incub_years"], "cat_cols": ["stage_in"], "interact_num_cat": [("incub_years", "stage_in")]},
    "rev_high":     {"bin_cols": ["team_size_in"],              "num_cols": ["incub_years"], "cat_cols": [],           "interact_num_cat": []},
    "team_growth":  {"bin_cols": ["tech_digital", "stage_adv"], "num_cols": [],             "cat_cols": [],           "interact_num_cat": []},
    "stage_growth": {"bin_cols": ["stage_adv", "team_size_in"], "num_cols": [],             "cat_cols": [],           "interact_num_cat": []},
}

PRIMARY   = "rev_high"
ALL_OBJS  = list(OUTCOME_SPECS.keys())
SECONDARY = [o for o in ALL_OBJS if o != PRIMARY]
print("Outcome specs:", list(OUTCOME_SPECS.keys()))

## Section 2 — Data Preparation

Load the cleaned dataset produced by `data_preparation.py` (or the companion
data-preparation notebook). The expected schema is documented in
`data/venture_cohort_dictionary.json`.

**Binary outcomes** (0/1):
- `ip` — intellectual property registered (patent/software) during or after graduation
- `hub` — post-graduation affiliation with hub/park/innovation space
- `rev_high` — high-revenue bracket (≥ R$5M–R$10M)
- `team_growth` — team headcount bracket grew after graduation
- `stage_growth` — technology maturity stage advanced after graduation

**Predictors**:
- `tech_digital` (0/1): digital/platform/Industry-4.0 tech domain
- `founders` (int): number of founding partners
- `incub_years` (float): incubation duration in years
- `stage_in` (1–4): technology maturity stage at incubation entry
- `team_size_in` (0/1): entry team size bracket (0=<5, 1=5–10)
- `stage_adv` (0/1): advanced entry stage (stage_in > 2)


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset: {len(df)} ventures × {df.shape[1]} columns")
print(f"\nOutcome prevalences:")
for col in ALL_OBJS:
    if col in df.columns:
        p = pd.to_numeric(df[col], errors="coerce").mean()
        print(f"  {col:12s}: {p:.3f}  ({int(round(p*len(df)))}/{len(df)})")
df.head()

## Section 3 — Helper FunctionsAll methodological steps are encapsulated here. These functions are calledonce per outcome in Section 4 and are fully reusable for other datasets.

In [ ]:
def build_design_matrix(df, spec):
    """
    Build standardized design matrix X and response vector y for a single outcome.

    Supports:
      bin_cols         : binary 0/1 predictors (kept as-is)
      num_cols         : continuous/ordinal predictors (standardized)
      cat_cols         : nominal predictors (one-hot, drop_first=True)
      interact_num_cat : list of (num_col, cat_col) — adds num_std * each cat dummy

    Returns
    -------
    X               : np.ndarray (n, p)
    y               : np.ndarray (n,)
    scaler          : fitted StandardScaler on num_cols
    num_used        : list[str] — continuous column names in X
    bin_used        : list[str] — binary column names in X
    cat_dummies     : list[str] — one-hot dummy column names in X
    interact_cols   : list[str] — interaction column names in X  (e.g. "incub_years:stage_in_2")
    interact_info   : list[dict] — each dict has keys: interact_col, num_col, cat_dummy
    """
    target           = spec["target"]
    bin_cols         = spec.get("bin_cols", [])
    num_cols         = spec.get("num_cols", [])
    cat_cols         = spec.get("cat_cols", [])
    interact_num_cat = spec.get("interact_num_cat", [])

    parts, num_used, bin_used, cat_dummies = [], [], [], []
    num_arrays  = {}   # col -> standardized 1-D array (for interaction construction)
    dummy_arrays = {}  # dummy_col -> 1-D array

    scaler = StandardScaler()
    if num_cols:
        Xn = df[num_cols].apply(pd.to_numeric, errors="coerce")
        Xn_s = scaler.fit_transform(Xn)
        parts.append(Xn_s)
        num_used = num_cols[:]
        for j, col in enumerate(num_cols):
            num_arrays[col] = Xn_s[:, j]

    if bin_cols:
        Xb = df[bin_cols].apply(pd.to_numeric, errors="coerce").values
        parts.append(Xb)
        bin_used = bin_cols[:]

    for col in cat_cols:
        dummies = pd.get_dummies(df[col].astype(str), drop_first=True,
                                 dtype=float, prefix=col)
        parts.append(dummies.values)
        for k, dc in enumerate(dummies.columns):
            cat_dummies.append(dc)
            dummy_arrays[dc] = dummies.values[:, k]

    # Interaction: z_num * d_k  for each (num_col, cat_col) pair
    interact_cols = []
    interact_info = []
    for num_col, cat_col in interact_num_cat:
        if num_col not in num_arrays:
            continue
        z_num = num_arrays[num_col]
        for dc in [c for c in cat_dummies if c.startswith(cat_col + "_")]:
            ic_name = f"{num_col}:{dc}"
            parts.append((z_num * dummy_arrays[dc]).reshape(-1, 1))
            interact_cols.append(ic_name)
            interact_info.append({"interact_col": ic_name,
                                   "num_col": num_col,
                                   "cat_dummy": dc})

    X = np.hstack(parts).astype(float)
    y = pd.to_numeric(df[target], errors="coerce").values.astype(float)
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    return X[mask], y[mask], scaler, num_used, bin_used, cat_dummies, interact_cols, interact_info

In [ ]:
def fit_loocv_ridge(X, y, C_grid):
    """
    Select ridge penalty C by leave-one-out cross-validation (log-loss criterion).

    Returns best_C (float) and grid_df (DataFrame with C vs loocv_logloss).
    """
    loo = LeaveOneOut()
    grid = []
    for C in C_grid:
        clf = LogisticRegression(l1_ratio=0, C=C, solver="lbfgs", max_iter=5000)
        y_true, y_pred = [], []
        try:
            for tr, te in loo.split(X):
                clf.fit(X[tr], y[tr])
                y_pred.append(clf.predict_proba(X[te])[:, 1][0])
                y_true.append(y[te][0])
            ll = log_loss(y_true, y_pred, labels=[0, 1])
        except Exception:
            ll = np.inf
        grid.append((C, ll))
    grid_df = pd.DataFrame(grid, columns=["C", "loocv_logloss"])
    best_C = float(grid_df.loc[grid_df["loocv_logloss"].idxmin(), "C"])
    return best_C, grid_df

In [ ]:
def _tune_C_inner(X_tr, y_tr, C_grid_inner, seed):
    """Select ridge C on (n-1) samples via StratifiedKFold (balanced weights)."""
    minority = int(np.bincount(y_tr.astype(int)).min())
    n_splits = max(3, min(5, minority))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    best_C, best_ll = C_grid_inner[0], np.inf
    for C in C_grid_inner:
        lls = []
        for tr_i, va_i in skf.split(X_tr, y_tr):
            clf_i = LogisticRegression(
                l1_ratio=0, C=C, solver="lbfgs",
                max_iter=2000, class_weight="balanced",
            )
            try:
                clf_i.fit(X_tr[tr_i], y_tr[tr_i])
                p_va = clf_i.predict_proba(X_tr[va_i])[:, 1]
                lls.append(log_loss(y_tr[va_i], p_va, labels=[0, 1]))
            except Exception:
                lls.append(np.inf)
        m = float(np.mean(lls))
        if m < best_ll:
            best_ll, best_C = m, C
    return best_C


def compute_loocv_metrics_and_permtest(X, y, best_C, B=2_000, seed=42,
                                        checkpoint_file=None):
    """
    Compute LOOCV metrics (AUC, LogLoss, Brier) and permutation p-values.

    Uses nested-CV approach matching the original script:
      * Within each LOOCV fold, C is re-selected via StratifiedKFold on n-1 samples.
      * class_weight="balanced" in every fold model.
      * Permutation test re-runs full nested LOOCV for each of B permuted labels.

    Checkpoint / resume
    -------------------
    If checkpoint_file (Path) is provided the permutation null distributions and
    RNG state are saved to disk every SAVE_EVERY draws.  On restart, execution
    resumes from the last saved draw — no permutation is repeated or skipped.

    Parameters
    ----------
    X               : np.ndarray
    y               : np.ndarray
    best_C          : float  — used only for post-hoc calibration, not inside LOOCV
    B               : int    — permutation draws
    seed            : int    — base RNG seed
    checkpoint_file : Path or None

    Returns
    -------
    metrics, calib  : dicts
    """
    import pickle as _pickle

    C_INNER   = np.logspace(-3, 3, 13)
    SAVE_EVERY = 200
    loo = LeaveOneOut()
    rng = np.random.default_rng(seed)

    # ── Nested LOOCV helper ───────────────────────────────────────────────────
    def _loocv_probs(X_, y_, base_seed):
        yt, yp = [], []
        for fold_idx, (tr, te) in enumerate(loo.split(X_)):
            fold_seed = base_seed + fold_idx
            C_fold = _tune_C_inner(X_[tr], y_[tr], C_INNER, seed=fold_seed)
            clf_f  = LogisticRegression(
                l1_ratio=0, C=C_fold, solver="lbfgs",
                max_iter=2000, class_weight="balanced",
            )
            try:
                clf_f.fit(X_[tr], y_[tr])
                yp.append(clf_f.predict_proba(X_[te])[:, 1][0])
            except Exception:
                yp.append(float(y_.mean()))
            yt.append(y_[te][0])
        return np.array(yt), np.array(yp)

    # ── Observed LOOCV ────────────────────────────────────────────────────────
    y_true, y_prob = _loocv_probs(X, y, base_seed=seed * 13)
    obs_auc = roc_auc_score(y_true, y_prob)
    obs_ll  = log_loss(y_true, y_prob, labels=[0, 1])
    obs_br  = brier_score_loss(y_true, y_prob)

    # ── Permutation null — with checkpoint support ────────────────────────────
    start_b      = 0
    null_auc     = []
    null_ll      = []
    null_br      = []

    if checkpoint_file is not None and Path(checkpoint_file).exists():
        try:
            with open(checkpoint_file, "rb") as _f:
                _ckpt = _pickle.load(_f)
            null_auc = _ckpt["null_auc"]
            null_ll  = _ckpt["null_ll"]
            null_br  = _ckpt["null_br"]
            start_b  = _ckpt["n_completed"]
            rng.bit_generator.state = _ckpt["rng_state"]
            print(f"  [RESUME] Permutation checkpoint: {start_b}/{B} draws loaded")
        except Exception as _e:
            print(f"  [WARN] Could not load permutation checkpoint ({_e}); starting from 0")
            start_b, null_auc, null_ll, null_br = 0, [], [], []

    for b in range(start_b, B):
        y_perm = rng.permutation(y)
        try:
            yt_b, yh_b = _loocv_probs(X, y_perm, base_seed=77 + b)
            if len(np.unique(y_perm)) == 2:
                null_auc.append(roc_auc_score(y_perm, yh_b))
            null_ll.append(log_loss(y_perm, yh_b, labels=[0, 1]))
            null_br.append(brier_score_loss(y_perm, yh_b))
        except Exception:
            pass

        # Save checkpoint every SAVE_EVERY draws
        if checkpoint_file is not None and (b + 1) % SAVE_EVERY == 0:
            _ckpt = {
                "null_auc":    null_auc,
                "null_ll":     null_ll,
                "null_br":     null_br,
                "n_completed": b + 1,
                "rng_state":   rng.bit_generator.state,
            }
            with open(checkpoint_file, "wb") as _f:
                _pickle.dump(_ckpt, _f)
            print(f"  [CHECKPOINT] {b + 1}/{B} permutations saved")

    # Remove checkpoint file on successful completion
    if checkpoint_file is not None and Path(checkpoint_file).exists():
        Path(checkpoint_file).unlink()
        print(f"  [CHECKPOINT] Permutation checkpoint removed (run complete)")

    def pval_hi(obs, null):
        null = np.array(null); null = null[np.isfinite(null)]
        return (1 + np.sum(null >= obs)) / (len(null) + 1) if len(null) > 0 else np.nan
    def pval_lo(obs, null):
        null = np.array(null); null = null[np.isfinite(null)]
        return (1 + np.sum(null <= obs)) / (len(null) + 1) if len(null) > 0 else np.nan

    metrics = {
        "AUC":    obs_auc, "p_AUC":   pval_hi(obs_auc, null_auc),
        "LogLoss": obs_ll,  "p_LL":    pval_lo(obs_ll,  null_ll),
        "Brier":  obs_br,  "p_Brier": pval_lo(obs_br,  null_br),
        "y_true": y_true,  "y_prob":  y_prob,
    }

    # ── Calibration ───────────────────────────────────────────────────────────
    p_bar = y_true.mean()
    uncertainty = p_bar * (1 - p_bar)
    n_bins = max(3, len(y_true) // 5)
    frac_pos, mean_pred = calibration_curve(y_true, y_prob,
                                            n_bins=n_bins, strategy="quantile")
    n_actual_bins = len(frac_pos)
    counts      = np.histogram(y_prob, bins=n_actual_bins)[0]
    reliability = np.average((mean_pred - frac_pos) ** 2, weights=counts + 1e-9)
    resolution  = np.average((frac_pos - p_bar) ** 2,  weights=counts + 1e-9)

    isotonic_probs   = None
    is_miscalibrated = reliability > 0.02
    if is_miscalibrated:
        base_clf = LogisticRegression(
            l1_ratio=0, C=best_C, solver="lbfgs", max_iter=5000,
        )
        minority = int(np.bincount(y.astype(int)).min())
        cal_clf  = CalibratedClassifierCV(
            base_clf, method="isotonic",
            cv=StratifiedKFold(
                n_splits=max(2, min(5, minority)),
                shuffle=True, random_state=42,
            ),
        )
        cal_clf.fit(X, y)
        isotonic_probs = cal_clf.predict_proba(X)[:, 1]

    calib = {
        "reliability": reliability, "resolution": resolution,
        "uncertainty": uncertainty, "is_miscalibrated": is_miscalibrated,
        "frac_pos": frac_pos, "mean_pred": mean_pred,
        "isotonic_probs": isotonic_probs,
    }
    return metrics, calib

In [ ]:
def compute_ames_with_bootstrap(clf, X, y, num_cols_used, bin_cols_used, scaler,
                               cat_dummies=None, interact_info=None,
                               R=2_000, seed=123):
    """
    Compute Average Marginal Effects (AMEs) with pairs-bootstrap 95%% CIs.

    Continuous predictors: derivative-based AME = mean[ p(1-p) * (beta_j + interaction_contrib) / sd_j ]
      where interaction_contrib sums beta_int_k * d_ki over any interaction terms for that predictor.
    Binary predictors: average discrete change P(Y=1|x=1) - P(Y=1|x=0).
    Categorical dummies: discrete change that also flips the corresponding interaction column.

    Returns ame_df with columns: [var, type, AME, CI_2.5, CI_97.5].
    """
    cat_dummies  = cat_dummies  or []
    interact_info = interact_info or []

    n_num = len(num_cols_used)
    n_bin = len(bin_cols_used)
    n_cat = len(cat_dummies)

    int_col_names = [d["interact_col"] for d in interact_info]

    def _num_idx(col):   return num_cols_used.index(col)
    def _bin_idx(col):   return n_num + bin_cols_used.index(col)
    def _cat_idx(col):   return n_num + n_bin + cat_dummies.index(col)
    def _int_idx(name):  return n_num + n_bin + n_cat + int_col_names.index(name)

    def _ames_once(clf_, X_, scaler_=None):
        mu = clf_.predict_proba(X_)[:, 1]
        b  = clf_.coef_.ravel()
        rows = []

        # Continuous — derivative AME accounting for interaction terms
        for j, col in enumerate(num_cols_used):
            sd = scaler_.scale_[j] if (scaler_ is not None and scaler_.scale_ is not None) else 1.0
            interact_contrib = np.zeros(len(X_))
            for idict in interact_info:
                if idict["num_col"] == col:
                    cat_j = _cat_idx(idict["cat_dummy"])
                    ic_j  = _int_idx(idict["interact_col"])
                    interact_contrib += b[ic_j] * X_[:, cat_j]
            dmu = mu * (1 - mu) * (b[j] + interact_contrib) / sd
            rows.append((col, "continuous", float(np.mean(dmu))))

        # Binary — standard discrete change
        for k, col in enumerate(bin_cols_used):
            j = _bin_idx(col)
            X0, X1 = X_.copy(), X_.copy()
            X0[:, j] = 0.0
            X1[:, j] = 1.0
            rows.append((col, "binary",
                         float(np.mean(clf_.predict_proba(X1)[:, 1]
                                      - clf_.predict_proba(X0)[:, 1]))))

        # Categorical dummies — discrete change that also updates interaction columns
        for col in cat_dummies:
            cat_j = _cat_idx(col)
            X0, X1 = X_.copy(), X_.copy()
            X0[:, cat_j] = 0.0
            X1[:, cat_j] = 1.0
            for idict in interact_info:
                if idict["cat_dummy"] == col:
                    ic_j  = _int_idx(idict["interact_col"])
                    num_j = _num_idx(idict["num_col"])
                    X0[:, ic_j] = 0.0
                    X1[:, ic_j] = X_[:, num_j]   # z_num_i when d_k flips to 1
            rows.append((col, "binary",
                         float(np.mean(clf_.predict_proba(X1)[:, 1]
                                      - clf_.predict_proba(X0)[:, 1]))))
        return rows

    ame_rows = _ames_once(clf, X, scaler)
    ame_df   = pd.DataFrame(ame_rows, columns=["var", "type", "AME"])

    rng  = np.random.default_rng(seed)
    boot = {r[0]: [] for r in ame_rows}
    for _ in range(R):
        idx = rng.integers(0, len(X), size=len(X))
        Xb, yb = X[idx], y[idx]
        sc_b = StandardScaler().fit(Xb[:, :n_num]) if n_num > 0 else None
        try:
            clf_b = LogisticRegression(l1_ratio=0, C=clf.C, solver="lbfgs", max_iter=5000)
            clf_b.fit(Xb, yb)
            for var, _, val in _ames_once(clf_b, Xb, sc_b):
                boot[var].append(val)
        except Exception:
            pass

    ci_rows = []
    for var, draws in boot.items():
        if draws:
            ci_rows.append({"var": var,
                            "CI_2.5":  float(np.percentile(draws,  2.5)),
                            "CI_97.5": float(np.percentile(draws, 97.5))})
    ci_df  = pd.DataFrame(ci_rows).set_index("var")
    ame_df = ame_df.set_index("var").join(ci_df, how="left").reset_index()
    return ame_df

In [ ]:
def run_outcome_pipeline(target_col, df, C_grid=C_GRID, B_perm=B_PERM,
                         R_boot=R_BOOT, seed_perm=SEED_PERM, seed_boot=SEED_BOOT,
                         save_prefix=None, checkpoint_dir=None):
    """
    Full modeling pipeline for one binary outcome.

    Steps
    -----
    1. Build design matrix from OUTCOME_SPECS[target_col]
    2. Select ridge penalty by LOOCV log-loss (full dataset, no balanced weights)
    3. Fit final model  (l1_ratio=0, pure L2 ridge, no class weighting)
    4. LOOCV metrics + permutation test  (nested C selection, class_weight="balanced")
       Permutation draws are checkpointed to checkpoint_dir/{target_col}_perm.pkl
       every 200 draws so execution can resume after interruption.
    5. AMEs with bootstrap CIs  (interaction-aware for hub)
    6. Save CSVs to OUTPUTS_DIR

    Parameters
    ----------
    checkpoint_dir : Path or None — directory for permutation checkpoint files
    """
    import pickle as _pickle

    spec = dict(OUTCOME_SPECS[target_col])
    spec["target"] = target_col

    (X, y, scaler,
     num_used, bin_used, cat_dummies,
     interact_cols, interact_info) = build_design_matrix(df, spec)

    print(f"\n{chr(9472)*55}")
    print(f"Outcome: {target_col.upper()}  |  n={len(y)}, p0={y.mean():.3f}")
    print(f"  Predictors — num:{num_used}  bin:{bin_used}  cat:{cat_dummies}")
    if interact_cols:
        print(f"  Interactions: {interact_cols}")

    # 1) LOOCV penalty selection (full dataset, unweighted)
    best_C, grid_df = fit_loocv_ridge(X, y, C_grid)
    print(f"  Best C={best_C:.4f} (LOOCV log-loss={grid_df['loocv_logloss'].min():.4f})")

    # 2) Final model (no class weighting — matches original final-model fit)
    clf = LogisticRegression(l1_ratio=0, C=best_C, solver="lbfgs", max_iter=5000)
    clf.fit(X, y)

    # 3) LOOCV metrics + permutation (nested CV, balanced weights, checkpointed)
    perm_ckpt = None
    if checkpoint_dir is not None:
        Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
        perm_ckpt = Path(checkpoint_dir) / f"{target_col}_perm.pkl"

    metrics, calib = compute_loocv_metrics_and_permtest(
        X, y, best_C, B=B_perm, seed=seed_perm,
        checkpoint_file=perm_ckpt,
    )
    print(f"  AUC={metrics['AUC']:.4f} (p={metrics['p_AUC']:.4f}) | "
          f"LL={metrics['LogLoss']:.4f} (p={metrics['p_LL']:.4f}) | "
          f"Brier={metrics['Brier']:.4f} (p={metrics['p_Brier']:.4f})")
    if calib["is_miscalibrated"]:
        print(f"  Miscalibration detected (reliability={calib['reliability']:.4f}); "
              f"isotonic post-calibration applied.")

    # 4) AMEs with bootstrap
    ame_df = compute_ames_with_bootstrap(
        clf, X, y, num_used, bin_used, scaler,
        cat_dummies=cat_dummies,
        interact_info=interact_info,
        R=R_boot, seed=seed_boot,
    )

    # 5) Save CSVs
    if save_prefix:
        pfx = OUTPUTS_DIR / save_prefix
        ame_df.to_csv(f"{pfx}_ames.csv", index=False)
        pd.DataFrame([{k: v for k, v in metrics.items()
                       if k not in ("y_true", "y_prob")}
                     ]).to_csv(f"{pfx}_loocv_perm.csv", index=False)

    return dict(metrics=metrics, calib=calib, ame_df=ame_df,
                clf=clf, X=X, y=y, scaler=scaler,
                num_used=num_used, bin_used=bin_used,
                cat_dummies=cat_dummies,
                interact_cols=interact_cols,
                interact_info=interact_info,
                spec=spec)

print("Helper functions defined.")

## Section 4 — Stage 1: Predictive ModelingRun the full pipeline for each outcome. Results are stored in `RESULTS` dict.

In [ ]:
import pickle as _pickle

# Checkpoint directories
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Outcomes to process : {ALL_OBJS}")
print(f"Permutation draws   : {B_PERM}  (saved every 200)")
print()

RESULTS = {}

for outcome in ALL_OBJS:
    outcome_ckpt = CHECKPOINT_DIR / f"{outcome}_result.pkl"

    # ── Load from outcome-level checkpoint if available ───────────────────────
    if outcome_ckpt.exists():
        try:
            with open(outcome_ckpt, "rb") as _f:
                RESULTS[outcome] = _pickle.load(_f)
            m = RESULTS[outcome]["metrics"]
            print(f"\n{chr(9472)*55}")
            print(f"[CHECKPOINT] {outcome.upper()} loaded from disk — skipping recomputation")
            print(f"  AUC={m['AUC']:.4f} (p={m['p_AUC']:.4f}) | "
                  f"LL={m['LogLoss']:.4f} (p={m['p_LL']:.4f}) | "
                  f"Brier={m['Brier']:.4f} (p={m['p_Brier']:.4f})")
            print(f"  AMEs: {RESULTS[outcome]['ame_df']['var'].tolist()}")
            continue
        except Exception as _e:
            print(f"[WARN] Could not load checkpoint for {outcome} ({_e}); recomputing")

    # ── Run full pipeline (permutations auto-checkpointed every 200 draws) ────
    RESULTS[outcome] = run_outcome_pipeline(
        target_col     = outcome,
        df             = df,
        save_prefix    = outcome,
        checkpoint_dir = CHECKPOINT_DIR,
    )

    # ── Save outcome-level checkpoint ─────────────────────────────────────────
    with open(outcome_ckpt, "wb") as _f:
        _pickle.dump(RESULTS[outcome], _f)
    print(f"  [CHECKPOINT] {outcome}_result.pkl saved")

print("\n\nAll outcomes processed.")

In [ ]:
# ── Table: LOOCV metrics and permutation p-values (Table 2 in paper) ─────────
rows = []
for outcome, res in RESULTS.items():
    m = res["metrics"]
    rows.append({"Outcome": outcome,
                 "AUC": m["AUC"],  "p_AUC": m["p_AUC"],
                 "LogLoss": m["LogLoss"], "p_LL": m["p_LL"],
                 "Brier": m["Brier"],   "p_Brier": m["p_Brier"]})

pred_table = pd.DataFrame(rows).set_index("Outcome").round(4)
pred_table.to_csv(OUTPUTS_DIR / "table_loocv_metrics.csv")
print("LOOCV metrics table:")
print(pred_table)

In [ ]:
# ── Table: AMEs with bootstrap CIs (Table 3 in paper) ────────────────────────
ame_tables = {}
for outcome, res in RESULTS.items():
    ame_df = res["ame_df"].copy()
    ame_df["Outcome"] = outcome
    ame_tables[outcome] = ame_df

all_ames = pd.concat(ame_tables.values(), ignore_index=True)
all_ames.to_csv(OUTPUTS_DIR / "table_ames.csv", index=False)
print("AME table (all outcomes):")
print(all_ames.to_string(index=False))

In [ ]:
# ── Calibration reliability curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, len(ALL_OBJS), figsize=(4*len(ALL_OBJS), 4), sharey=True)
for ax, outcome in zip(axes, ALL_OBJS):
    calib = RESULTS[outcome]["calib"]
    ax.plot(calib["mean_pred"], calib["frac_pos"], "s-", label="LOOCV probs")
    if calib["isotonic_probs"] is not None:
        y_true = RESULTS[outcome]["metrics"]["y_true"]
        frac_c, mean_c = calibration_curve(y_true, calib["isotonic_probs"],
                                           n_bins=5, strategy="quantile")
        ax.plot(mean_c, frac_c, "^--", label="Isotonic")
    ax.plot([0,1],[0,1],"k--", alpha=0.4, label="Perfect")
    ax.set_title(outcome); ax.set_xlabel("Mean predicted"); ax.legend(fontsize=7)
axes[0].set_ylabel("Fraction positives")
fig.suptitle("Reliability (Calibration) Curves — LOOCV predictions", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "fig_calibration_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Calibration curves saved.")

## Section 5 — Per-Venture Objective CoefficientsFor each venture *i* and outcome *j*:$$g_{ij} = p_{0j} + \sum_k \text{AME}_{jk} \cdot x_{ik}$$where $p_{0j}$ is the sample prevalence and AME medians are used as point estimates.These coefficients serve as the objective weights $u_{ij}$ in the MILP.

In [ ]:
def build_objective_coeffs(df, ame_df, target_col, scaler=None, num_used=None, bin_used=None):
    """
    Compute per-venture objective coefficient g_i = p0 + sum_k AME_k * x_ik.

    For one-hot dummy columns (e.g. "stage_in_2") that are not in df directly,
    the dummy is reconstructed from the base categorical column.

    Returns a Series indexed by df.index with g_i values in [0, 1] after min-max norm.
    """
    y_vec = pd.to_numeric(df[target_col], errors="coerce")
    p0    = float(y_vec.mean())

    ame_map = dict(zip(ame_df["var"], ame_df["AME"]))
    g = np.full(len(df), p0, dtype=float)

    for var, ame_val in ame_map.items():
        if var in df.columns:
            # Direct column (binary or numeric predictor)
            x_col = pd.to_numeric(df[var], errors="coerce").fillna(0).values
            g += ame_val * x_col
        else:
            # Try to reconstruct as a one-hot dummy: "base_col_level"
            reconstructed = False
            for base_col in df.columns:
                if var.startswith(base_col + "_"):
                    level = var[len(base_col) + 1:]
                    dummy = (df[base_col].astype(str) == level).astype(float).values
                    g += ame_val * dummy
                    reconstructed = True
                    break
            if not reconstructed:
                pass   # silently skip interaction-derived columns

    return pd.Series(g, index=df.index, name=target_col)

df_coeffs = {"startup_id": np.arange(1, len(df)+1)}
for outcome, res in RESULTS.items():
    g = build_objective_coeffs(df, res["ame_df"], outcome,
                               scaler=res["scaler"], num_used=res["num_used"],
                               bin_used=res["bin_used"])
    df_coeffs[outcome] = g.values

COEFFS = pd.DataFrame(df_coeffs)

# Min-max normalize to [0,1] per objective
NORM = COEFFS.copy()
for col in ALL_OBJS:
    lo, hi = COEFFS[col].min(), COEFFS[col].max()
    NORM[col + "_01"] = (COEFFS[col] - lo) / (hi - lo) if hi > lo else 0.0

NORM.to_csv(OUTPUTS_DIR / "objectives_normalized.csv", index=False)
print("Per-venture coefficients computed and normalized.")
print(NORM[[c + "_01" for c in ALL_OBJS]].describe().round(3))

## Section 6 — Stage 2: ε-Constraint Portfolio OptimizationSolves the capacity-constrained MO-BIP:$$\max_{x \in \{0,1\}^n} \sum_i u_{i,\text{primary}} x_i$$$$\text{s.t.} \quad \sum_i x_i = K, \quad \sum_i u_{ij} x_i \geq \varepsilon_j \; \forall j \neq \text{primary}$$**Evidence-weighted variant** (optional): scale each $u_{ij}$ by a confidencefactor derived from permutation p-values before optimization:$$\tilde{u}_{ij} = u_{ij} \cdot w_j, \quad w_j = 1 - p_j^{\text{perm}}$$Set `USE_EVIDENCE_WEIGHTS=True` to activate.

In [ ]:
# ── Evidence-weighting (optional) ─────────────────────────────────────────────
USE_EVIDENCE_WEIGHTS = False   # set True for the evidence-weighted variant

obj_cols_norm = [c + "_01" for c in ALL_OBJS]

if USE_EVIDENCE_WEIGHTS:
    # Weight each objective by (1 - permutation p-value for AUC)
    ew = {}
    for col, outcome in zip(obj_cols_norm, ALL_OBJS):
        p_auc = RESULTS[outcome]["metrics"]["p_AUC"]
        ew[col] = max(0.0, 1.0 - p_auc)
    print("Evidence weights:", {k: round(v, 3) for k, v in ew.items()})
else:
    ew = {c: 1.0 for c in obj_cols_norm}
    print("Evidence weights: uniform (1.0 per objective)")

In [ ]:
import itertools as _itertools

def _topk_upper(DF_norm, col, K):
    """Sum of the K largest normalized values for col — upper bound on ε floor."""
    vals = DF_norm[col].sort_values(ascending=False).values
    return float(np.sum(vals[:K])) if len(vals) >= K else float(np.sum(vals))


def _dominance_mask(points):
    """True if point is non-dominated (maximisation in all dimensions)."""
    n = points.shape[0]
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]:
            continue
        pi = points[i]
        for j in range(n):
            if i == j or not keep[j]:
                continue
            pj = points[j]
            if np.all(pj >= pi) and np.any(pj > pi):
                keep[i] = False
                break
    return keep


def _hypervolume_2d(points, ref=(0.0, 0.0)):
    """
    2D hypervolume (maximisation) with reference point ref.
    Filters to non-dominated set, sorts by x, integrates swept area.
    """
    if points.size == 0:
        return 0.0
    pts = points[_dominance_mask(points)]
    if pts.shape[0] == 0:
        return 0.0
    pts = pts[np.argsort(pts[:, 0])]
    hv, x_prev, y_best = 0.0, ref[0], ref[1]
    for x, y in pts:
        hv    += max(0.0, x - x_prev) * max(0.0, y - ref[1])
        x_prev = x
        y_best = max(y_best, y)
    return float(max(0.0, hv))


def solve_eps_constraint(DF_norm, primary_col, secondary_cols,
                          K, eps_grid=None, eps_points=6,
                          obj_weights=None, verbose=False):
    """
    Sweep the ε-constraint MILP.

    Grid strategy (matching original script):
      For each secondary, build an independent ε-grid from 0 to its top-K upper
      bound with eps_points values.  The full Cartesian product (eps_points ^ n_sec
      instances) is solved and deduplicated on solution signature (objective sums).

    For backward-compatible single-grid usage, pass eps_grid (1-D array) and
    eps_points is ignored; the same ε is applied uniformly to all secondaries.

    Parameters
    ----------
    eps_grid   : 1-D array or None.  If None, per-secondary grids are used.
    eps_points : int — points per secondary grid (default 6 → 6^4=1296 instances).

    Returns
    -------
    frontier : DataFrame  [eps_<sec>, status, gap, solve_time, obj_value,
                            selected, *secondary_sums]
               Deduplicated on solution signature; Coverage_eps = feas / total.
    """
    if obj_weights is None:
        obj_weights = {c: 1.0 for c in [primary_col] + secondary_cols}

    ids = DF_norm["startup_id"].astype(int).tolist()

    # Build per-secondary ε grids
    uniform_mode = eps_grid is not None
    if uniform_mode:
        # Uniform mode: same ε for every secondary
        combos_raw = list(eps_grid)
        combos = [dict(zip(secondary_cols, [eps] * len(secondary_cols)))
                  for eps in combos_raw]
    else:
        sec_grids = {}
        for sec in secondary_cols:
            ub = _topk_upper(DF_norm, sec, K)
            sec_grids[sec] = np.linspace(0.0, ub, eps_points).tolist()
        combos = [dict(zip(secondary_cols, vals))
                  for vals in _itertools.product(*[sec_grids[s] for s in secondary_cols])]

    rows = []
    for eps_floors in combos:
        prob = pulp.LpProblem("eps_constraint", pulp.LpMaximize)
        x    = {i: pulp.LpVariable(f"x_{i}", cat="Binary") for i in ids}

        prob += pulp.lpSum(
            obj_weights.get(primary_col, 1.0) *
            float(DF_norm.loc[DF_norm["startup_id"]==i, primary_col].values[0]) * x[i]
            for i in ids)
        prob += (pulp.lpSum(x[i] for i in ids) == K), "capacity"
        for sec, floor_val in eps_floors.items():
            prob += (pulp.lpSum(
                obj_weights.get(sec, 1.0) *
                float(DF_norm.loc[DF_norm["startup_id"]==i, sec].values[0]) * x[i]
                for i in ids) >= floor_val), f"floor_{sec}"

        prob.solve(pulp.PULP_CBC_CMD(msg=int(verbose)))

        status     = pulp.LpStatus[prob.status]
        solve_time = float(prob.solutionTime)
        gap        = 0.0 if status == "Optimal" else float("nan")
        obj_val    = pulp.value(prob.objective) if status == "Optimal" else float("nan")
        selected   = ([i for i in ids if pulp.value(x[i]) and pulp.value(x[i]) > 0.5]
                      if status == "Optimal" else [])
        sec_sums   = {}
        for sec in secondary_cols:
            if status == "Optimal":
                sec_sums[sec] = sum(
                    float(DF_norm.loc[DF_norm["startup_id"]==i, sec].values[0])
                    for i in selected)
            else:
                sec_sums[sec] = float("nan")

        # In uniform mode add single "eps" column for MC loop compatibility
        if uniform_mode:
            common_eps = eps_floors[secondary_cols[0]]
            eps_cols = {"eps": common_eps,
                        **{f"eps_{s}": v for s, v in eps_floors.items()}}
        else:
            eps_cols = {f"eps_{s}": v for s, v in eps_floors.items()}
        row = {**eps_cols,
               "status": status, "gap": gap, "solve_time": solve_time,
               "obj_value": obj_val, "selected": selected, **sec_sums}
        rows.append(row)

    df = pd.DataFrame(rows)

    # Deduplicate on solution signature (same portfolio → same objective sums)
    sig_cols = [sec for sec in secondary_cols] + ["obj_value"]
    df_dedup = df.drop_duplicates(subset=sig_cols, keep="first").reset_index(drop=True)
    n_raw, n_dedup = len(df), len(df_dedup)
    feas_raw = (df["status"] == "Optimal").sum()
    print(f"  Instances: {n_raw} | Feasible: {feas_raw} ({feas_raw/n_raw:.1%}) | "
          f"Unique after dedup: {n_dedup}")
    return df_dedup

print("MILP solver function defined.")

In [ ]:
# ── Run ε-constraint sweep ────────────────────────────────────────────────────
PRIMARY_COL   = PRIMARY + "_01"
SECONDARY_COLS= [c + "_01" for c in SECONDARY]

frontier = solve_eps_constraint(
    DF_norm        = NORM,
    primary_col    = PRIMARY_COL,
    secondary_cols = SECONDARY_COLS,
    K              = CAPACITY,
    eps_points     = EPS_POINTS,   # per-secondary grid: EPS_POINTS^4 instances
    obj_weights    = ew,
)

feasible  = frontier[frontier["status"]=="Optimal"]
n_total   = len(frontier)
n_feasible= len(feasible)
print(f"Instances: {n_total} | Feasible: {n_feasible} ({n_feasible/n_total:.1%})")
print(f"Optimality gap: always 0.00 for feasible instances (CBC exact solver)")
frontier.to_csv(OUTPUTS_DIR / "frontier.csv", index=False)
frontier.head(8)

In [ ]:
# ── Pareto front scatter (Figure: front_scatter) ─────────────────────────────
feasible = frontier[frontier["status"] == "Optimal"].copy()
if len(feasible) > 0:
    y = feasible["obj_value"].to_numpy(dtype=float)
    x = feasible[SECONDARY_COLS].mean(axis=1).to_numpy(dtype=float)

    def _mm(a):
        lo, hi = np.min(a), np.max(a)
        return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

    x_n, y_n = _mm(x), _mm(y)

    plt.figure(figsize=(5.6, 4.2))
    plt.scatter(x_n, y_n, s=28, alpha=0.7, edgecolors="none")
    plt.xlabel("Mean of secondaries (min–max normalized on frontier)")
    plt.ylabel("Primary — rev_high (min–max normalized on frontier)")
    plt.title("Pareto Front — ε-constraint sweep")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / "front_scatter.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"front_scatter.png saved  ({len(feasible)} feasible points)")

## Section 7 — Stage 3: Monte Carlo Uncertainty PropagationPropagates uncertainty in AMEs (sampled from Normal distributions implied bybootstrap CIs) and baseline rates (Beta priors) across `MC_DRAWS` draws.For each draw, the ε-constraint problem is solved for a policy scenario comparingtwo candidate options (*Expand* an incumbent vs. *Admit* a waitlist entrant)across a grid of α (Expand gain scaling) and ε (secondary floor) values.**Outputs**: selection probabilities, feasibility surfaces, Pareto front bands,switching thresholds, and robust rankings.

In [ ]:
# ── AME uncertainty sampling ──────────────────────────────────────────────────
def sample_ames(RESULTS, rng):
    """
    Sample one draw of AMEs per outcome from Normal(median, se_approx).
    se_approx = (CI_97.5 - CI_2.5) / (2 * 1.96)
    """
    sampled = {}
    for outcome, res in RESULTS.items():
        ame_df = res["ame_df"].copy()
        vals = []
        for _, row in ame_df.iterrows():
            lo, hi = row.get("CI_2.5", row["AME"]), row.get("CI_97.5", row["AME"])
            se = (hi - lo) / (2 * 1.96) if (hi - lo) > 0 else 1e-6
            vals.append(rng.normal(loc=row["AME"], scale=se))
        sampled[outcome] = pd.DataFrame({"var": ame_df["var"].values, "AME": vals})
    return sampled

# ── Baseline rate sampling ─────────────────────────────────────────────────────
def sample_baselines(df, ALL_OBJS, rng):
    """Sample baseline prevalences p0 from Beta(a, b) priors."""
    p0s = {}
    for col in ALL_OBJS:
        y_  = pd.to_numeric(df[col], errors="coerce").dropna()
        a, b_ = float(y_.sum()) + 0.5, float((1-y_).sum()) + 0.5
        p0s[col] = float(rng.beta(a, b_))
    return p0s

print("Monte Carlo sampling functions defined.")

In [ ]:
# ── Define candidate options (Expand vs Admit) ───────────────────────────────
# CUSTOMIZE: define the venture profiles for your policy scenario.
# Each entry is a dict mapping predictor variables to their values for that venture.
# Incumbent ventures are already incubated; waitlisted ventures are candidates.
# The "alpha" parameter scales the effective gain credited to the Expand option
# (alpha=1.0 means expansion benefits equal admission benefits).

INCUMBENTS  = ["I1", "I2", "I7"]          # labels of incumbent ventures
WAITLISTED  = ["W2", "W3"]                # labels of waitlisted ventures

# ── Load observed cohort for baseline probability estimation ──────────────────
obs_df = df.copy()
obs_df["startup_id"] = np.arange(1, len(obs_df)+1)

print(f"Policy scenario: Expand ({INCUMBENTS}) vs Admit ({WAITLISTED})")
print(f"MC draws: {MC_DRAWS}, alpha grid: {ALPHAS}, eps grid length: {len(EPS_VEC)}")

In [ ]:
# ── Monte Carlo main loop ─────────────────────────────────────────────────────
rng_mc  = np.random.default_rng(SEED_MC)
log_rows = []

for draw in range(MC_DRAWS):
    sampled_ames = sample_ames(RESULTS, rng_mc)
    p0s          = sample_baselines(obs_df, ALL_OBJS, rng_mc)

    # Build per-startup coefficients from this draw's AMEs and baselines
    g_draw = {"startup_id": obs_df["startup_id"].tolist()}
    for outcome in ALL_OBJS:
        ame_df_draw = sampled_ames[outcome]
        g = build_objective_coeffs(obs_df, ame_df_draw, outcome)
        g = g - g.mean() + p0s[outcome]          # re-center to sampled baseline
        g_draw[outcome] = np.clip(g.values, 0, 1)

    DF_draw = pd.DataFrame(g_draw)

    # Normalize
    DF_draw_norm = DF_draw.copy()
    for col in ALL_OBJS:
        lo, hi = DF_draw[col].min(), DF_draw[col].max()
        DF_draw_norm[col+"_01"] = (DF_draw[col]-lo)/(hi-lo) if hi>lo else 0.0

    for alpha in ALPHAS:
        # Scale Expand candidates by alpha
        DF_scenario = DF_draw_norm.copy()
        for col in [c+"_01" for c in ALL_OBJS]:
            expand_mask = DF_scenario["startup_id"].isin(
                [i for i, lbl in enumerate(INCUMBENTS, 1) if True])
            DF_scenario.loc[expand_mask, col] *= alpha

        frontier_draw = solve_eps_constraint(
            DF_norm=DF_scenario, primary_col=PRIMARY+"_01",
            secondary_cols=[c+"_01" for c in SECONDARY],
            K=CAPACITY, eps_grid=EPS_VEC, verbose=False)

        for _, row in frontier_draw.iterrows():
            log_rows.append({"draw": draw, "alpha": alpha, "eps": row["eps"],
                             "status": row["status"], "obj_value": row.get("obj_value"),
                             "selected": str(row.get("selected",""))})

    if (draw+1) % 200 == 0:
        print(f"  MC draw {draw+1}/{MC_DRAWS}")

log_df = pd.DataFrame(log_rows)
log_df.to_csv(OUTPUTS_DIR / "mc_selection_log.csv", index=False)
print(f"Monte Carlo complete. {len(log_df)} rows logged.")

In [ ]:
# ── MC Summary: P(Feasible) and P(Expand) heatmaps ───────────────────────────
agg = (log_df.groupby(["alpha","eps"])
       .apply(lambda g: pd.Series({
           "P_feasible": (g["status"]=="Optimal").mean(),
           "P_expand":   (g["status"]=="Optimal").mean(),   # refine with your labeling
           "rev_high":    g["obj_value"].mean(skipna=True),
       })).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes,
                           ["P_feasible", "rev_high"],
                           ["P(Feasible)", "Mean primary obj (RevHigh)"]):
    pivot = agg.pivot(index="alpha", columns="eps", values=col)
    im = ax.imshow(pivot.values, aspect="auto", origin="lower",
                   extent=[EPS_VEC.min(), EPS_VEC.max(),
                           min(ALPHAS)-.1, max(ALPHAS)+.1])
    plt.colorbar(im, ax=ax)
    ax.set_xlabel("ε"); ax.set_ylabel("α"); ax.set_title(title)
plt.suptitle("Monte Carlo — Policy Scenario Summary")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "mc_heat_expand.png", dpi=150, bbox_inches="tight")
plt.show()
print("MC heatmaps saved.")

In [ ]:
# ── MC Summary: Switching thresholds ─────────────────────────────────────────
threshold_rows = []
for alpha, g in log_df[log_df["status"]=="Optimal"].groupby("alpha"):
    switches = []
    for draw_id, dg in g.groupby("draw"):
        dg = dg.sort_values("eps")
        # Detect first ε where dominant option switches (simplified)
        switches.append(float(dg["eps"].min()))  # placeholder: customize per scenario
    threshold_rows.append({
        "alpha": alpha,
        "eps_switch_median": np.median(switches),
        "eps_switch_p25":    np.percentile(switches, 25),
        "eps_switch_p75":    np.percentile(switches, 75),
        "coverage":          len(switches)/MC_DRAWS,
    })

thresh_df = pd.DataFrame(threshold_rows)
thresh_df.to_csv(OUTPUTS_DIR / "mc_thresholds.csv", index=False)
print("Switching thresholds:")
print(thresh_df.round(3).to_string(index=False))

## Section 8 — MILP Diagnostics

In [ ]:
# ── MILP diagnostics and front quality — Table 6 and Table 7 ─────────────────

feasible = frontier[frontier["status"] == "Optimal"].copy()
feas_mask = frontier["status"] == "Optimal"

n_total    = len(frontier)
n_feasible = int(feas_mask.sum())
feas_rate  = n_feasible / n_total if n_total > 0 else float("nan")
opt_rate   = 1.0   # CBC certifies optimality on every feasible instance

median_gap   = float(frontier.loc[feas_mask, "gap"].median())   if feas_mask.any() else float("nan")
median_stime = float(frontier.loc[feas_mask, "solve_time"].median()) if feas_mask.any() else float("nan")
p95_stime    = float(frontier.loc[feas_mask, "solve_time"].quantile(0.95)) if feas_mask.any() else float("nan")

# Secondary mean column for plots / HV
sec_mean_col = "sec_mean"
frontier[sec_mean_col] = frontier[SECONDARY_COLS].mean(axis=1)
if len(feasible) > 0:
    feasible[sec_mean_col] = feasible[SECONDARY_COLS].mean(axis=1)

# ── Front quality ─────────────────────────────────────────────────────────────
if len(feasible) > 0:
    y_raw = feasible["obj_value"].to_numpy(dtype=float)
    x_raw = feasible[sec_mean_col].to_numpy(dtype=float)

    def _mm(a):
        lo, hi = np.min(a), np.max(a)
        return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

    x_n, y_n = _mm(x_raw), _mm(y_raw)
    pts = np.c_[x_n, y_n]

    # True 2D hypervolume (with reference at origin, after normalization)
    hv_2d        = _hypervolume_2d(pts, ref=(0.0, 0.0))
    # Spread = std of normalized primary (matches original front_quality)
    spread_primary = float(np.std(y_n))
    # ND share via dominance mask
    nd_share       = float(np.mean(_dominance_mask(pts)))
    cardinality    = int(n_feasible)
    coverage_eps   = feas_rate
else:
    hv_2d = spread_primary = nd_share = coverage_eps = float("nan")
    cardinality = 0

# ── Print ─────────────────────────────────────────────────────────────────────
print("=" * 58)
print("TABLE 6 — MILP diagnostics (CBC)")
print(f"  Total instances (unique) : {n_total}")
print(f"  P(Feasible)              : {feas_rate:.3f}")
print(f"  P(Optimal | Feasible)    : {opt_rate:.3f}")
print(f"  Median relative gap      : {median_gap:.4f}")
print(f"  Median solve time (s)    : {median_stime:.4f}")
print(f"  95th-pct solve time (s)  : {p95_stime:.4f}")
print()
print("TABLE 7 — Pareto front quality")
print(f"  HV_2D                    : {hv_2d:.3f}")
print(f"  Spread(Primary) [std]    : {spread_primary:.3f}")
print(f"  Cardinality              : {cardinality}")
print(f"  ND Share                 : {nd_share:.3f}")
print(f"  Coverage_ε               : {coverage_eps:.3f}")
print("=" * 58)

# ── Save CSVs ─────────────────────────────────────────────────────────────────
pd.DataFrame([{
    "n_total": n_total, "n_feasible": n_feasible,
    "P_feasible": round(feas_rate, 4),
    "P_optimal_given_feasible": round(opt_rate, 4),
    "median_gap": round(median_gap, 4),
    "median_solve_time_s": round(median_stime, 4),
    "p95_solve_time_s": round(p95_stime, 4),
}]).to_csv(OUTPUTS_DIR / "table6_milp_diagnostics.csv", index=False)

pd.DataFrame([{
    "HV_2D": round(hv_2d, 4),
    "spread_primary_std": round(spread_primary, 4),
    "cardinality": cardinality,
    "nd_share": round(nd_share, 4),
    "coverage_eps": round(coverage_eps, 4),
}]).to_csv(OUTPUTS_DIR / "table7_front_quality.csv", index=False)

frontier.loc[feas_mask, [c for c in frontier.columns
                          if c.startswith("eps_")] + ["solve_time", "gap"]].to_csv(
    OUTPUTS_DIR / "milp_solve_times.csv", index=False)

print("Saved: table6_milp_diagnostics.csv | table7_front_quality.csv | milp_solve_times.csv")

In [ ]:
# ── ECDF of solve times — Figure 1 in paper ──────────────────────────────────
feas_mask = frontier["status"] == "Optimal"

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

# (a) Solve time ECDF
times = frontier.loc[feas_mask, "solve_time"].to_numpy(dtype=float)
if len(times) > 0:
    t_sorted = np.sort(times)
    F_t = np.arange(1, len(t_sorted) + 1) / len(t_sorted)
    axes[0].plot(t_sorted, F_t, lw=2)
    axes[0].set_xlabel("Solve time (s)")
    axes[0].set_ylabel("ECDF")
    axes[0].set_title("Empirical CDF of solve times")
    axes[0].grid(True, alpha=0.25)

# (b) Gap ECDF (all feasible — gap = 0.0, but shown for completeness)
gaps = frontier.loc[feas_mask, "gap"].dropna().to_numpy(dtype=float)
if len(gaps) > 0:
    g_sorted = np.sort(gaps)
    F_g = np.arange(1, len(g_sorted) + 1) / len(g_sorted)
    axes[1].plot(g_sorted, F_g, lw=2)
    axes[1].set_xlabel("CBC MIP gap")
    axes[1].set_ylabel("ECDF")
    axes[1].set_title("Empirical CDF of MIP gaps")
    axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "ecdf_time.png", dpi=200, bbox_inches="tight")
plt.show()
print("ecdf_time.png saved")

## Section 9 — Export Summary

In [ ]:
print("Files written to", OUTPUTS_DIR)
for f in sorted(OUTPUTS_DIR.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:45s} {size:>8,} bytes")